# Multiple GA Results Analysis & DNA Pruning

This notebook:
1. Searches through GA result folders for all `aggregated_results.pkl` files
2. Finds all DNA vectors exceeding a score threshold
3. Runs weight pruning on each high-scoring DNA
4. Creates interactive plots with slider to browse between different DNA vectors

In [30]:
# Import required libraries
import os
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.offline as pyo
from ipywidgets import interact, IntSlider, Dropdown, VBox, HBox, Output, Button
from IPython.display import display, clear_output
import time
from copy import deepcopy

# Import project modules
from src.constants import *
from src.neuron import *
from src.network import *
from src.validation import *
from src.genetic_algorithm import *

# Import weight pruning functionality
import sys
sys.path.append('.')
from weight_pruning import WeightPruner, evaluate_single_dna

print("✅ All imports successful")

✅ All imports successful


## Configuration

In [31]:
# Configuration
RESULTS_FOLDER = "results/multiple_runs_H_opt4_20250815_180350"  # Change this to your results folder
SCORE_THRESHOLD = 965  # Minimum score threshold for DNA selection
PRUNING_THRESHOLD = 970  # Minimum score to maintain during pruning
MAX_DNAS_TO_PROCESS = 3000  # Limit number of DNAs to prevent overwhelming

# Duplicate removal options (Step 1)
REMOVE_EXACT_DUPLICATES = True  # Remove DNAs with identical vectors
UNIQUE_CONFIGURATIONS_ONLY = True  # Keep only best DNA for each unique non-zero pattern

# Post-pruning filtering options (Step 2)
POST_PRUNING_UNIQUE_CONFIGS = True  # Apply unique configuration filtering after pruning
MAX_PRUNED_WEIGHTS = 18  # Maximum non-zero weights allowed in pruned DNA (None = no limit)

print(f"Configuration:")
print(f"  Results folder: {RESULTS_FOLDER}")
print(f"  Score threshold: {SCORE_THRESHOLD}")
print(f"  Pruning threshold: {PRUNING_THRESHOLD}")
print(f"  Max DNAs to process: {MAX_DNAS_TO_PROCESS}")
print(f"  Remove exact duplicates: {REMOVE_EXACT_DUPLICATES}")
print(f"  Unique configurations only: {UNIQUE_CONFIGURATIONS_ONLY}")
print(f"  Post-pruning unique configs: {POST_PRUNING_UNIQUE_CONFIGS}")
print(f"  Max pruned weights: {MAX_PRUNED_WEIGHTS or 'No limit'}")

Configuration:
  Results folder: results/multiple_runs_H_opt4_20250815_180350
  Score threshold: 965
  Pruning threshold: 970
  Max DNAs to process: 3000
  Remove exact duplicates: True
  Unique configurations only: True
  Post-pruning unique configs: True
  Max pruned weights: 18


## Step 1: Find All High-Scoring DNA Vectors

In [32]:
def find_all_aggregated_results(results_folder):
    """Find all aggregated_results.pkl files in the results folder and subfolders."""
    results_path = Path(results_folder)
    aggregated_files = []
    
    if not results_path.exists():
        print(f"❌ Results folder does not exist: {results_folder}")
        return []
    
    # Recursively search for aggregated_results.pkl files
    for pkl_file in results_path.rglob("aggregated_results.pkl"):
        aggregated_files.append(pkl_file)
    
    print(f"📁 Found {len(aggregated_files)} aggregated_results.pkl files")
    for f in aggregated_files:
        print(f"  {f}")
    
    return aggregated_files

def remove_exact_duplicates(high_scoring_dnas):
    """Remove DNAs with identical vectors."""
    print(f"🔍 Removing exact duplicates from {len(high_scoring_dnas)} DNAs...")
    
    unique_dnas = []
    seen_vectors = set()
    
    for dna_info in high_scoring_dnas:
        # Create a hashable representation of the DNA vector
        dna_tuple = tuple(dna_info['dna'])
        
        if dna_tuple not in seen_vectors:
            seen_vectors.add(dna_tuple)
            unique_dnas.append(dna_info)
    
    duplicates_removed = len(high_scoring_dnas) - len(unique_dnas)
    print(f"  ✅ Removed {duplicates_removed} exact duplicates, {len(unique_dnas)} unique DNAs remain")
    
    return unique_dnas

def filter_unique_configurations(high_scoring_dnas):
    """Keep only the best DNA for each unique non-zero weight pattern."""
    print(f"🎯 Filtering for unique configurations from {len(high_scoring_dnas)} DNAs...")
    
    # Group DNAs by their non-zero weight pattern (mask)
    configuration_groups = {}
    
    for dna_info in high_scoring_dnas:
        # Create a binary mask showing which weights are non-zero
        mask = tuple(1 if x != 0 else 0 for x in dna_info['dna'])
        
        if mask not in configuration_groups:
            configuration_groups[mask] = []
        
        configuration_groups[mask].append(dna_info)
    
    print(f"  Found {len(configuration_groups)} unique weight configurations:")
    
    # For each configuration, keep only the highest-scoring DNA
    unique_config_dnas = []
    
    for i, (mask, dnas_in_group) in enumerate(configuration_groups.items()):
        # Sort by score (highest first) and take the best
        best_dna = max(dnas_in_group, key=lambda x: x['total_score'])
        unique_config_dnas.append(best_dna)
        
        non_zero_count = sum(mask)
        group_scores = [d['total_score'] for d in dnas_in_group]
        
        print(f"    Config {i+1}: {non_zero_count} non-zero weights, "
              f"{len(dnas_in_group)} DNAs (scores: {min(group_scores)}-{max(group_scores)}), "
              f"kept best: {best_dna['total_score']}")
    
    # Sort by score (highest first)
    unique_config_dnas.sort(key=lambda x: x['total_score'], reverse=True)
    
    configurations_removed = len(high_scoring_dnas) - len(unique_config_dnas)
    print(f"  ✅ Removed {configurations_removed} duplicate configurations, "
          f"{len(unique_config_dnas)} unique configurations remain")
    
    return unique_config_dnas

def extract_high_scoring_dnas(aggregated_files, score_threshold):
    """Extract all DNA vectors that exceed the score threshold."""
    high_scoring_dnas = []
    
    for file_path in aggregated_files:
        try:
            with open(file_path, 'rb') as f:
                data = pickle.load(f)
            
            run_folder = file_path.parent.name
            
            # Extract high-scoring DNAs from this file
            for dna_record in data.get('all_dna_tested', []):
                total_score = dna_record.get('total_score', 0)
                
                if total_score >= score_threshold:
                    high_scoring_dnas.append({
                        'dna': dna_record['dna'].copy(),
                        'total_score': total_score,
                        'exp_score': dna_record['exp_score'],
                        'cont_score': dna_record['cont_score'],
                        'generation': dna_record['generation'],
                        'process_id': dna_record['process_id'],
                        'individual_id': dna_record['individual_id'],
                        'run_folder': run_folder,
                        'source_file': str(file_path),
                        'non_zero_weights': int(np.count_nonzero(dna_record['dna']))
                    })
        
        except Exception as e:
            print(f"⚠️  Error reading {file_path}: {e}")
    
    # Sort by score (highest first)
    high_scoring_dnas.sort(key=lambda x: x['total_score'], reverse=True)
    
    print(f"\n🎯 Found {len(high_scoring_dnas)} DNA vectors with score >= {score_threshold}")
    
    # Apply duplicate removal if requested
    if REMOVE_EXACT_DUPLICATES and high_scoring_dnas:
        high_scoring_dnas = remove_exact_duplicates(high_scoring_dnas)
    
    # Apply unique configuration filtering if requested
    if UNIQUE_CONFIGURATIONS_ONLY and high_scoring_dnas:
        high_scoring_dnas = filter_unique_configurations(high_scoring_dnas)
    
    # Show summary
    if high_scoring_dnas:
        best_score = high_scoring_dnas[0]['total_score']
        worst_score = high_scoring_dnas[-1]['total_score']
        avg_score = np.mean([d['total_score'] for d in high_scoring_dnas])
        
        print(f"\n📊 Final dataset summary:")
        print(f"  Score range: {worst_score} - {best_score}")
        print(f"  Average score: {avg_score:.1f}")
        print(f"  Non-zero weights range: {min(d['non_zero_weights'] for d in high_scoring_dnas)} - {max(d['non_zero_weights'] for d in high_scoring_dnas)}")
        
        # Show top 5
        print(f"\n🏆 Top 5 DNA vectors:")
        for i, dna in enumerate(high_scoring_dnas[:5]):
            print(f"  {i+1}. Score: {dna['total_score']} (Exp:{dna['exp_score']}, Cont:{dna['cont_score']}) "
                  f"Weights:{dna['non_zero_weights']}, Gen:{dna['generation']}, Run: {dna['run_folder']}")
    
    return high_scoring_dnas

# Execute step 1
print("🔍 Step 1: Finding all high-scoring DNA vectors...")
aggregated_files = find_all_aggregated_results(RESULTS_FOLDER)
high_scoring_dnas = extract_high_scoring_dnas(aggregated_files, SCORE_THRESHOLD)

# Limit number of DNAs to process
if len(high_scoring_dnas) > MAX_DNAS_TO_PROCESS:
    print(f"\n⚠️  Found {len(high_scoring_dnas)} DNAs, limiting to top {MAX_DNAS_TO_PROCESS} for performance")
    high_scoring_dnas = high_scoring_dnas[:MAX_DNAS_TO_PROCESS]

print(f"\n✅ Step 1 complete: {len(high_scoring_dnas)} DNA vectors ready for pruning")

🔍 Step 1: Finding all high-scoring DNA vectors...
📁 Found 6 aggregated_results.pkl files
  results/multiple_runs_H_opt4_20250815_180350/run_006/aggregated_results.pkl
  results/multiple_runs_H_opt4_20250815_180350/run_001/aggregated_results.pkl
  results/multiple_runs_H_opt4_20250815_180350/run_004/aggregated_results.pkl
  results/multiple_runs_H_opt4_20250815_180350/run_003/aggregated_results.pkl
  results/multiple_runs_H_opt4_20250815_180350/run_002/aggregated_results.pkl
  results/multiple_runs_H_opt4_20250815_180350/run_005/aggregated_results.pkl

🎯 Found 22215 DNA vectors with score >= 965
🔍 Removing exact duplicates from 22215 DNAs...
  ✅ Removed 222 exact duplicates, 21993 unique DNAs remain
🎯 Filtering for unique configurations from 21993 DNAs...
  Found 418 unique weight configurations:
    Config 1: 47 non-zero weights, 806 DNAs (scores: 965-975), kept best: 975
    Config 2: 45 non-zero weights, 62 DNAs (scores: 965-975), kept best: 975
    Config 3: 46 non-zero weights, 287

## Step 2: Prune Each High-Scoring DNA

In [ ]:
def prune_dna_vectors(high_scoring_dnas, pruning_threshold):
    """Run weight pruning on each high-scoring DNA vector."""
    pruned_results = []
    
    print(f"🔧 Starting pruning process for {len(high_scoring_dnas)} DNA vectors...")
    
    for i, dna_info in enumerate(high_scoring_dnas):
        print(f"\n🧬 Pruning DNA {i+1}/{len(high_scoring_dnas)} (Score: {dna_info['total_score']})...")
        
        try:
            # Initialize pruner
            pruner = WeightPruner(dna_info['dna'], pruning_threshold, tmax=5000)
            
            # Run greedy magnitude pruning (fastest method)
            pruned_dna = pruner.impact_based_pruning()
            
            # Evaluate pruned DNA
            final_exp, final_cont, final_total = evaluate_single_dna(pruned_dna, 5000)
            
            pruned_result = {
                'original_dna': dna_info,
                'pruned_dna': pruned_dna.copy(),
                'original_score': dna_info['total_score'],
                'pruned_score': final_total,
                'original_nonzero': np.count_nonzero(dna_info['dna']),
                'pruned_nonzero': np.count_nonzero(pruned_dna),
                'weights_removed': np.count_nonzero(dna_info['dna']) - np.count_nonzero(pruned_dna),
                'pruning_history': pruner.pruning_history,
                'final_exp_score': final_exp,
                'final_cont_score': final_cont,
                'id': i + 1
            }
            
            pruned_results.append(pruned_result)
            
            reduction_pct = (pruned_result['weights_removed'] / pruned_result['original_nonzero']) * 100
            print(f"  ✅ Success: {pruned_result['original_nonzero']} → {pruned_result['pruned_nonzero']} weights "
                  f"({reduction_pct:.1f}% reduction), Score: {pruned_result['original_score']} → {pruned_result['pruned_score']}")
            
        except Exception as e:
            print(f"  ❌ Error pruning DNA {i+1}: {e}")
            continue
    
    print(f"\n✅ Pruning complete: {len(pruned_results)} successfully pruned DNA vectors")
    
    # Summary statistics
    if pruned_results:
        avg_reduction = np.mean([r['weights_removed']/r['original_nonzero']*100 for r in pruned_results])
        total_original_weights = sum(r['original_nonzero'] for r in pruned_results)
        total_pruned_weights = sum(r['pruned_nonzero'] for r in pruned_results)
        
        print(f"\n📊 Pruning Summary:")
        print(f"  Average weight reduction: {avg_reduction:.1f}%")
        print(f"  Total weights: {total_original_weights} → {total_pruned_weights}")
        print(f"  Best pruned score: {max(r['pruned_score'] for r in pruned_results)}")
        print(f"  Most efficient (fewest weights): {min(r['pruned_nonzero'] for r in pruned_results)} weights")
    
    # Filter by maximum pruned weights if specified
    if MAX_PRUNED_WEIGHTS is not None and pruned_results:
        print(f"\n⚖️  Filtering pruned DNAs by maximum weight count ({MAX_PRUNED_WEIGHTS} weights)...")
        
        pre_filter_count = len(pruned_results)
        pruned_results = [r for r in pruned_results if r['pruned_nonzero'] <= MAX_PRUNED_WEIGHTS]
        filtered_count = pre_filter_count - len(pruned_results)
        
        print(f"  ✅ Filtered out {filtered_count} DNAs with > {MAX_PRUNED_WEIGHTS} weights, "
              f"{len(pruned_results)} DNAs remain")
        
        if pruned_results:
            weight_counts = [r['pruned_nonzero'] for r in pruned_results]
            print(f"  Remaining weight range: {min(weight_counts)} - {max(weight_counts)} weights")
        else:
            print(f"  ⚠️  No DNAs have ≤ {MAX_PRUNED_WEIGHTS} weights after pruning")
    
    # Apply post-pruning unique configuration filtering if requested
    if POST_PRUNING_UNIQUE_CONFIGS and pruned_results:
        print(f"\n🎯 Post-pruning: Filtering for unique configurations from {len(pruned_results)} pruned DNAs...")
        
        # Group pruned DNAs by their non-zero weight pattern (mask)
        configuration_groups = {}
        
        for pruned_result in pruned_results:
            # Create a binary mask showing which weights are non-zero in the pruned DNA
            mask = tuple(1 if x != 0 else 0 for x in pruned_result['pruned_dna'])
            
            if mask not in configuration_groups:
                configuration_groups[mask] = []
            
            configuration_groups[mask].append(pruned_result)
        
        print(f"  Found {len(configuration_groups)} unique pruned weight configurations:")
        
        # For each configuration, keep only the highest-scoring pruned DNA
        unique_config_pruned = []
        
        for i, (mask, pruned_dnas_in_group) in enumerate(configuration_groups.items()):
            # Sort by pruned score (highest first) and take the best
            best_pruned = max(pruned_dnas_in_group, key=lambda x: x['pruned_score'])
            unique_config_pruned.append(best_pruned)
            
            non_zero_count = sum(mask)
            group_scores = [d['pruned_score'] for d in pruned_dnas_in_group]
            
            print(f"    Config {i+1}: {non_zero_count} non-zero weights, "
                  f"{len(pruned_dnas_in_group)} pruned DNAs (scores: {min(group_scores)}-{max(group_scores)}), "
                  f"kept best: {best_pruned['pruned_score']}")
        
        # Sort by pruned score (highest first)
        unique_config_pruned.sort(key=lambda x: x['pruned_score'], reverse=True)
        
        configurations_removed = len(pruned_results) - len(unique_config_pruned)
        print(f"  ✅ Removed {configurations_removed} duplicate pruned configurations, "
              f"{len(unique_config_pruned)} unique pruned configurations remain")
        
        pruned_results = unique_config_pruned
    
    return pruned_results

# Execute step 2
if high_scoring_dnas:
    print("\n🔧 Step 2: Pruning all high-scoring DNA vectors...")
    pruned_results = prune_dna_vectors(high_scoring_dnas, PRUNING_THRESHOLD)
    
    # Apply final limit after all filtering
    if len(pruned_results) > MAX_DNAS_TO_PROCESS:
        print(f"\n⚠️  After pruning and filtering: {len(pruned_results)} DNAs, limiting to top {MAX_DNAS_TO_PROCESS} for performance")
        pruned_results = pruned_results[:MAX_DNAS_TO_PROCESS]
    
    print(f"\n✅ Step 2 complete: {len(pruned_results)} pruned DNA vectors ready for visualization")
else:
    print("❌ No high-scoring DNAs found. Check your results folder and score threshold.")
    pruned_results = []


🔧 Step 2: Pruning all high-scoring DNA vectors...
🔧 Starting pruning process for 418 DNA vectors...

🧬 Pruning DNA 1/418 (Score: 975)...
Original DNA evaluation:
  Exp: 476, Cont: 499, Total: 975
  Non-zero weights: 47
  Zero weights: 6

🎯 Starting impact-based pruning (threshold: 970)
  ✅ Removed weight 1 (value:   7) -> Score: 975 (Exp: 476, Cont: 499), Non-zero: 46
  ✅ Removed weight 4 (value: -89) -> Score: 975 (Exp: 476, Cont: 499), Non-zero: 45
  ✅ Removed weight 5 (value: -38) -> Score: 975 (Exp: 476, Cont: 499), Non-zero: 44
  ✅ Removed weight 6 (value:  -6) -> Score: 975 (Exp: 476, Cont: 499), Non-zero: 43
  ✅ Removed weight 9 (value:  -2) -> Score: 975 (Exp: 476, Cont: 499), Non-zero: 42
  ✅ Removed weight 13 (value:  -5) -> Score: 975 (Exp: 476, Cont: 499), Non-zero: 41
  ✅ Removed weight 14 (value:  -2) -> Score: 975 (Exp: 476, Cont: 499), Non-zero: 40
  ✅ Removed weight 15 (value:  -3) -> Score: 975 (Exp: 476, Cont: 499), Non-zero: 39
  ✅ Removed weight 22 (value:   1) ->

## Step 3: Create Interactive Visualization Functions

In [ ]:
import networkx as nx
from matplotlib.patches import FancyArrowPatch
import matplotlib.patheffects as pe

def create_directed_graph_from_dna(dna_array, dna_info):
    """Create a directed graph visualization from a DNA vector."""
    
    # Define custom positions for each node (from current_workbook.ipynb)
    neu_coords = {
        'Somat': (5, 8.5),
        'MSN1': (5, 5.5),
        'MSN2': (3, 5.5),
        'MSN3': (0, 5.5),
        'SNR1': (5, 2.5),
        'SNR2': (3, 2.5),
        'SNR3': (0, 2.5),
        'ALMinter': (2, 8.5),
        'PPN': (2.5, 0),
        'THALgo': (2, 4.5),
        'VMprep': (4, 1),
        'ALMprep': (4, 7),
        'ALMresp': (1, 7),
        'VMresp': (1, 1)
    }
    
    # Create directed graph and add all neuron nodes
    G = nx.DiGraph()
    G.add_nodes_from(NEURON_NAMES)
    
    # Extract non-zero connections from DNA
    connections = []
    
    for i, weight in enumerate(dna_array):
        if weight != 0 and i < len(ACTIVE_SYNAPSES):
            from_neuron, to_neuron = ACTIVE_SYNAPSES[i]
            
            # Determine connection type
            is_inhibitory = from_neuron in INHIBITORY_NEURONS
            style = 'dashed' if is_inhibitory else 'solid'
            
            # Determine edge width based on absolute weight
            abs_weight = abs(int(weight))
            if abs_weight >= 300:
                width = 4
            elif abs_weight >= 100:
                width = 3
            elif abs_weight >= 50:
                width = 2
            else:
                width = 1
            
            # Add edge with attributes
            G.add_edge(from_neuron, to_neuron, 
                      weight=f'{int(weight)}', 
                      style=style, 
                      width=width,
                      raw_weight=int(weight))
            
            connections.append((from_neuron, to_neuron, int(weight)))
    
    return G, neu_coords, connections

def create_network_plot(G, neu_coords, connections, dna_info):
    """Create network graph plot for a single DNA."""
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Separate edges into reciprocal and non-reciprocal
    reciprocal_edges_pairs = set()
    non_reciprocal_edges_data = []
    all_edges_with_data = list(G.edges(data=True))
    
    # Check for reciprocal edges
    for u, v, data in all_edges_with_data:
        if G.has_edge(v, u):  # Reciprocal edge exists
            reciprocal_edges_pairs.add(tuple(sorted([u, v])))
        else:  # Non-reciprocal edge
            non_reciprocal_edges_data.append((u, v, data))
    
    # Draw nodes with uniform color
    nx.draw_networkx_nodes(G, pos=neu_coords, 
                          node_size=3000, 
                          node_color='lightblue', 
                          edgecolors='black',
                          linewidths=2,
                          ax=ax)
    
    # Draw node labels
    nx.draw_networkx_labels(G, pos=neu_coords, 
                           font_size=10, 
                           font_weight='bold', 
                           font_color='black',
                           ax=ax)
    
    # Draw non-reciprocal edges with improved arrow visibility
    if non_reciprocal_edges_data:
        edgelist_nr = [(u, v) for u, v, data in non_reciprocal_edges_data]
        styles_nr = [data['style'] for u, v, data in non_reciprocal_edges_data]
        widths_nr = [data['width'] for u, v, data in non_reciprocal_edges_data]
        
        nx.draw_networkx_edges(G, pos=neu_coords, 
                              edgelist=edgelist_nr,
                              node_size=3000,
                              arrowstyle='-|>', 
                              arrowsize=30,  # Larger arrows
                              edge_color='black',
                              width=widths_nr, 
                              style=styles_nr,
                              connectionstyle='arc3,rad=0.02',  # Slight curve to avoid node centers
                              min_source_margin=15,  # Space from source node
                              min_target_margin=15,  # Space from target node
                              ax=ax)
    
    # Draw reciprocal edges manually with offset
    if reciprocal_edges_pairs:
        parallel_offset = 0.12  # Slightly larger offset
        shorten_length = 0.4   # More aggressive shortening
        
        for u_orig, v_orig in reciprocal_edges_pairs:
            pos_u = np.array(neu_coords[u_orig])
            pos_v = np.array(neu_coords[v_orig])
            
            vec = pos_v - pos_u
            vec_len = np.linalg.norm(vec)
            if vec_len < 1e-6:
                continue
            
            unit_vec = vec / vec_len
            perp_vec = np.array([-unit_vec[1], unit_vec[0]])
            
            # Shorten to avoid overlap with nodes
            start_u = pos_u + unit_vec * shorten_length
            end_v = pos_v - unit_vec * shorten_length
            start_v = pos_v + unit_vec * shorten_length  
            end_u = pos_u - unit_vec * shorten_length
            
            # Draw both directions
            for direction, (start_pos, end_pos, src, dst) in enumerate([
                (start_u + perp_vec * parallel_offset, end_v + perp_vec * parallel_offset, u_orig, v_orig),
                (start_v - perp_vec * parallel_offset, end_u - perp_vec * parallel_offset, v_orig, u_orig)
            ]):
                # Get edge data
                edge_data = G[src][dst]
                edge_style = edge_data['style']
                edge_width = edge_data['width']
                
                arrow = FancyArrowPatch(start_pos, end_pos,
                                       arrowstyle='-|>',
                                       shrinkA=0, shrinkB=0,
                                       mutation_scale=30,  # Larger arrow heads
                                       linewidth=edge_width,
                                       linestyle='--' if edge_style == 'dashed' else '-',
                                       color='black',
                                       alpha=0.8)
                ax.add_patch(arrow)
    
    # Add edge labels for non-reciprocal edges
    if non_reciprocal_edges_data:
        labels_nr = {}
        all_edge_labels = nx.get_edge_attributes(G, 'weight')
        
        for u, v, data in non_reciprocal_edges_data:
            edge_tuple = (u, v)
            if edge_tuple in all_edge_labels:
                labels_nr[edge_tuple] = all_edge_labels[edge_tuple]
        
        if labels_nr:
            nx.draw_networkx_edge_labels(G, pos=neu_coords, 
                                        edge_labels=labels_nr,
                                        label_pos=0.5, 
                                        rotate=True,
                                        font_color='black', 
                                        font_size=8, 
                                        bbox=dict(boxstyle='round,pad=0.2', 
                                                facecolor='white', 
                                                alpha=0.8),
                                        ax=ax)
    
    # Add edge labels for reciprocal edges manually
    if reciprocal_edges_pairs:
        label_perp_offset = parallel_offset * 1.4
        
        for u_orig, v_orig in reciprocal_edges_pairs:
            pos_u = np.array(neu_coords[u_orig])
            pos_v = np.array(neu_coords[v_orig])
            vec = pos_v - pos_u
            vec_len = np.linalg.norm(vec)
            if vec_len < 1e-6:
                continue
            
            unit_vec = vec / vec_len
            perp_vec = np.array([-unit_vec[1], unit_vec[0]])
            
            # Label positions
            mid_point = (pos_u + pos_v) / 2
            
            # Label for u -> v
            if G.has_edge(u_orig, v_orig):
                label_pos_uv = mid_point + perp_vec * label_perp_offset
                weight_uv = G[u_orig][v_orig]['weight']
                ax.text(label_pos_uv[0], label_pos_uv[1], weight_uv,
                       fontsize=8, ha='center', va='center',
                       bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
            
            # Label for v -> u
            if G.has_edge(v_orig, u_orig):
                label_pos_vu = mid_point - perp_vec * label_perp_offset
                weight_vu = G[v_orig][u_orig]['weight']
                ax.text(label_pos_vu[0], label_pos_vu[1], weight_vu,
                       fontsize=8, ha='center', va='center',
                       bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
    
    # Set title and formatting
    plt.title(f'DNA {dna_info["id"]} - Network Topology\n'
             f'Score: {dna_info["pruned_score"]} | '
             f'Connections: {dna_info["pruned_nonzero"]}/{dna_info["original_nonzero"]} | '
             f'Reduction: {dna_info["weights_removed"]/dna_info["original_nonzero"]*100:.1f}%\n'
             f'Run: {dna_info["original_dna"]["run_folder"]} | Gen: {dna_info["original_dna"]["generation"]}\n'
             f'Dashed lines = inhibitory connections',
             fontsize=14, fontweight='bold', pad=20)
    
    # Set axis properties
    ax.set_aspect('equal')
    plt.grid(True, alpha=0.3)
    plt.xlabel('Time, ms', fontsize=12)
    plt.ylabel('Voltage, mV', fontsize=12)
    
    plt.tight_layout()
    return fig

def run_dna_with_voltage_tracking(dna_array):
    """Run DNA simulation and capture voltage traces."""
    # Convert DNA to weight matrix
    dna_matrix = decode_dna_to_matrix(dna_array)
    
    # Create experimental setup
    all_neurons = create_neurons()
    splits, input_waves, alpha_array = create_experiment()
    cue_wave, go_wave = input_waves
    
    results = {}
    
    # Run both experimental and control conditions
    for label, ctl in [("experimental", False), ("control", True)]:
        # Prepare neurons for this condition
        prepare_neurons(all_neurons, cue_wave, go_wave, ctl)
        
        # Reset time pointer
        import src.neuron as neuron_module
        neuron_module.t_pointer = 0
        
        # Create voltage history tracking
        voltage_history = np.zeros((len(NEURON_NAMES), TMAX), dtype=np.float32)
        spike_history = np.zeros((len(NEURON_NAMES), TMAX), dtype=np.uint8)
        
        # Import voltage arrays
        from src.neuron import _V, _VHIST, _SPIKES, _INPUT, _vpeak
        from src.network import convert_to_sparse_weights, sparse_matrix_multiply_only_spiking, _sparse_connections
        
        # Store initial voltage
        voltage_history[:, 0] = _V.copy()
        
        # Run simulation
        N = len(all_neurons)
        spikers = np.zeros(N, np.uint8)
        sparse_weights = convert_to_sparse_weights(dna_matrix)
        
        for t in range(TMAX - 1):
            # Apply synaptic input
            if spikers.any():
                post_I = sparse_matrix_multiply_only_spiking(spikers.astype(np.float32), 
                                                            sparse_weights, 
                                                            _sparse_connections)
                lend = min(alpha_array.size, TMAX - t - 1)
                _INPUT[:, t+1:t+1+lend] += post_I[:, None] * alpha_array[:lend]
            
            # Take integration step
            spikers = vectorised_step(_INPUT[:, t])
            spike_history[:, t] = spikers
            
            # Record voltage
            if t > 0:
                peaked = spike_history[:, t-1].astype(bool)
                voltage_history[peaked, t-1] = _vpeak[peaked]
            
            voltage_history[:, t+1] = _V.copy()
        
        # Handle final timestep
        final_spikers = vectorised_step(_INPUT[:, TMAX-1])
        spike_history[:, TMAX-1] = final_spikers
        
        # Create voltage traces dictionary
        voltage_traces = {}
        for i, neuron_name in enumerate(NEURON_NAMES):
            voltage_traces[neuron_name] = voltage_history[i, :]
        
        # Calculate spike counts
        spike_counts = evaluate_conditions(spike_history)
        
        results[label] = {
            'voltages': voltage_traces,
            'spike_counts': spike_counts[label] if label in spike_counts else 0,
            'spikes': spike_history.copy()
        }
    
    return results

def create_voltage_plot(results, dna_info):
    """Create interactive voltage trace plot for a single DNA."""
    time_ms = np.arange(TMAX)
    n_neurons = len(NEURON_NAMES)
    
    subplot_titles = []
    for neuron_name in NEURON_NAMES:
        criteria_marker = " *" if neuron_name in CRITERIA_NAMES else ""
        subplot_titles.extend([f'{neuron_name}{criteria_marker} - Experimental', 
                             f'{neuron_name}{criteria_marker} - Control'])
    
    fig = make_subplots(
        rows=n_neurons, 
        cols=2,
        subplot_titles=subplot_titles,
        vertical_spacing=0.02,
        horizontal_spacing=0.08
    )
    
    # Add traces for each neuron
    for i, neuron_name in enumerate(NEURON_NAMES):
        row = i + 1
        is_criteria_neuron = neuron_name in CRITERIA_NAMES
        line_width = 2 if is_criteria_neuron else 1
        
        # Experimental condition
        voltages_exp = results['experimental']['voltages'][neuron_name]
        fig.add_trace(
            go.Scatter(
                x=time_ms,
                y=voltages_exp,
                mode='lines',
                name=f'{neuron_name} Exp',
                line=dict(color='blue', width=line_width),
                hovertemplate='<b>%{fullData.name}</b><br>' +
                             'Time: %{x} ms<br>' +
                             'Voltage: %{y:.2f} mV<br>' +
                             '<extra></extra>',
                showlegend=False
            ),
            row=row, col=1
        )
        
        # Control condition
        voltages_ctrl = results['control']['voltages'][neuron_name]
        fig.add_trace(
            go.Scatter(
                x=time_ms,
                y=voltages_ctrl,
                mode='lines',
                name=f'{neuron_name} Ctrl',
                line=dict(color='red', width=line_width),
                hovertemplate='<b>%{fullData.name}</b><br>' +
                             'Time: %{x} ms<br>' +
                             'Voltage: %{y:.2f} mV<br>' +
                             '<extra></extra>',
                showlegend=False
            ),
            row=row, col=2
        )
        
        # Add stimulus markers
        for col in [1, 2]:
            fig.add_vrect(
                x0=1000, x1=1200,
                fillcolor="red", opacity=0.2,
                layer="below", line_width=0,
                row=row, col=col
            )
            fig.add_vrect(
                x0=3000, x1=3100,
                fillcolor="green", opacity=0.2,
                layer="below", line_width=0,
                row=row, col=col
            )
    
    # Update layout
    fig.update_layout(
        title=f'DNA {dna_info["id"]} - Score: {dna_info["pruned_score"]} | ' + 
              f'Weights: {dna_info["original_nonzero"]}→{dna_info["pruned_nonzero"]} | ' +
              f'Run: {dna_info["original_dna"]["run_folder"]}' +
              f'<br><sub>* = Criteria neurons | Red=Cue | Green=Go</sub>',
        height=300 * n_neurons,
        width=1200,
        showlegend=False,
        hovermode='closest'
    )
    
    # Update y-axes with borders for criteria neurons
    for i in range(1, n_neurons + 1):
        neuron_name = NEURON_NAMES[i-1]
        is_criteria_neuron = neuron_name in CRITERIA_NAMES
        
        if is_criteria_neuron:
            border_style = dict(linewidth=3, linecolor='gold', mirror=True)
        else:
            border_style = dict(linewidth=1, linecolor='lightgray', mirror=True)
        
        fig.update_yaxes(range=[-100, 100], title_text="Voltage (mV)", row=i, col=1, **border_style)
        fig.update_yaxes(range=[-100, 100], title_text="Voltage (mV)", row=i, col=2, **border_style)
        fig.update_xaxes(row=i, col=1, **border_style)
        fig.update_xaxes(row=i, col=2, **border_style)
    
    # Update x-axes
    fig.update_xaxes(title_text="Time (ms)", row=n_neurons, col=1)
    fig.update_xaxes(title_text="Time (ms)", row=n_neurons, col=2)
    
    return fig

print("✅ Visualization functions defined")

✅ Visualization functions defined


## Step 4: Generate All Simulation Results

In [ ]:
def generate_all_simulation_results(pruned_results):
    """Pre-generate simulation results for all pruned DNAs."""
    simulation_results = []
    
    print(f"🧮 Generating simulation results for {len(pruned_results)} pruned DNAs...")
    
    for i, dna_info in enumerate(pruned_results):
        print(f"  Simulating DNA {i+1}/{len(pruned_results)}... ", end="")
        
        try:
            # Run simulation with pruned DNA
            results = run_dna_with_voltage_tracking(dna_info['pruned_dna'])
            simulation_results.append(results)
            print("✅")
            
        except Exception as e:
            print(f"❌ Error: {e}")
            simulation_results.append(None)
    
    print(f"\n✅ Simulation complete: {sum(1 for r in simulation_results if r is not None)} successful simulations")
    return simulation_results

# Generate all simulation results if we have pruned results
if pruned_results:
    print("\n🧮 Step 3: Generating simulation results for all pruned DNAs...")
    simulation_results = generate_all_simulation_results(pruned_results)
else:
    simulation_results = []
    print("❌ No pruned results available for simulation")


🧮 Step 3: Generating simulation results for all pruned DNAs...
🧮 Generating simulation results for 2 pruned DNAs...
  Simulating DNA 1/2... ✅
  Simulating DNA 2/2... ✅

✅ Simulation complete: 2 successful simulations


## Step 5: Interactive DNA Browser with Slider

In [33]:
def create_dual_dna_browser():
    """Create interactive DNA browser with both voltage plots and network graphs."""
    if not pruned_results or not simulation_results:
        print("❌ No data available for browsing")
        return
    
    # Create widgets
    dna_slider = IntSlider(
        value=0,
        min=0,
        max=len(pruned_results) - 1,
        step=1,
        description='DNA #:',
        style={'description_width': 'initial'},
        continuous_update=False
    )
    
    # Sort options
    sort_dropdown = Dropdown(
        options=[
            ('By Score (High→Low)', 'score_desc'),
            ('By Score (Low→High)', 'score_asc'),
            ('By Weights Removed (Most→Least)', 'removed_desc'),
            ('By Weights Removed (Least→Most)', 'removed_asc'),
            ('By Final Weight Count (Least→Most)', 'final_weights_asc'),
            ('By Final Weight Count (Most→Least)', 'final_weights_desc'),
            ('By Original Order', 'original')
        ],
        value='score_desc',
        description='Sort by:',
        style={'description_width': 'initial'}
    )
    
    # Visualization type selector
    viz_dropdown = Dropdown(
        options=[
            ('Voltage Traces', 'voltage'),
            ('Network Graph', 'network'),
            ('Both', 'both')
        ],
        value='both',
        description='Show:',
        style={'description_width': 'initial'}
    )
    
    output = Output()
    
    # Store sorted indices
    sorted_indices = list(range(len(pruned_results)))
    
    def sort_data(sort_by):
        nonlocal sorted_indices
        
        if sort_by == 'score_desc':
            sorted_indices = sorted(range(len(pruned_results)), 
                                  key=lambda i: pruned_results[i]['pruned_score'], reverse=True)
        elif sort_by == 'score_asc':
            sorted_indices = sorted(range(len(pruned_results)), 
                                  key=lambda i: pruned_results[i]['pruned_score'])
        elif sort_by == 'removed_desc':
            sorted_indices = sorted(range(len(pruned_results)), 
                                  key=lambda i: pruned_results[i]['weights_removed'], reverse=True)
        elif sort_by == 'removed_asc':
            sorted_indices = sorted(range(len(pruned_results)), 
                                  key=lambda i: pruned_results[i]['weights_removed'])
        elif sort_by == 'final_weights_asc':
            sorted_indices = sorted(range(len(pruned_results)), 
                                  key=lambda i: pruned_results[i]['pruned_nonzero'])
        elif sort_by == 'final_weights_desc':
            sorted_indices = sorted(range(len(pruned_results)), 
                                  key=lambda i: pruned_results[i]['pruned_nonzero'], reverse=True)
        else:  # original
            sorted_indices = list(range(len(pruned_results)))
        
        # Reset slider
        dna_slider.value = 0
    
    def update_plot(dna_index, sort_by, viz_type):
        with output:
            clear_output(wait=True)
            
            # Get actual index after sorting
            actual_index = sorted_indices[dna_index]
            
            dna_info = pruned_results[actual_index]
            sim_result = simulation_results[actual_index]
            
            if sim_result is None:
                print(f"❌ No simulation data available for DNA {actual_index + 1}")
                return
            
            # Display DNA information
            print(f"🧬 DNA {dna_index + 1} of {len(pruned_results)} (Original Index: {actual_index + 1})")
            print(f"📊 Scores: Original={dna_info['original_score']}, Pruned={dna_info['pruned_score']} "
                  f"(Exp:{dna_info['final_exp_score']}, Cont:{dna_info['final_cont_score']})")
            print(f"⚖️  Weights: {dna_info['original_nonzero']} → {dna_info['pruned_nonzero']} "
                  f"({dna_info['weights_removed']} removed, {dna_info['weights_removed']/dna_info['original_nonzero']*100:.1f}% reduction)")
            print(f"📁 Source: {dna_info['original_dna']['run_folder']}, Gen {dna_info['original_dna']['generation']}")
            
            print(f"\\n🧬 Pruned DNA Vector:")
            print(f"  {dna_info['pruned_dna']}")
            
            # Show network connections
            G, neu_coords, connections = create_directed_graph_from_dna(dna_info['pruned_dna'], dna_info)
            if connections:
                print(f"\\n🔗 Network Connections ({len(connections)} total):")
                for from_neuron, to_neuron, weight in connections:
                    is_inhibitory = from_neuron in INHIBITORY_NEURONS
                    conn_type = "Inhibitory" if is_inhibitory else "Excitatory"
                    print(f"  {from_neuron:8s} → {to_neuron:8s} | {weight:6d} | {conn_type}")
            else:
                print("\\n⚠️  No connections found in pruned DNA")
            
            # Create and display plots based on selection
            if viz_type in ['voltage', 'both']:
                print("\\n📈 Voltage Traces:")
                voltage_fig = create_voltage_plot(sim_result, dna_info)
                voltage_fig.show()
            
            if viz_type in ['network', 'both']:
                print("\\n🌐 Network Topology:")
                if connections:
                    network_fig = create_network_plot(G, neu_coords, connections, dna_info)
                    plt.show()
                else:
                    print("  ⚠️  Cannot create network graph: No connections to display")
    
    # Set up interactions
    def on_sort_change(change):
        sort_data(change['new'])
        update_plot(dna_slider.value, change['new'], viz_dropdown.value)
    
    def on_slider_change(change):
        update_plot(change['new'], sort_dropdown.value, viz_dropdown.value)
    
    def on_viz_change(change):
        update_plot(dna_slider.value, sort_dropdown.value, change['new'])
    
    sort_dropdown.observe(on_sort_change, names='value')
    dna_slider.observe(on_slider_change, names='value')
    viz_dropdown.observe(on_viz_change, names='value')
    
    # Initial sort
    sort_data(sort_dropdown.value)
    
    # Create layout
    controls = HBox([sort_dropdown, viz_dropdown, dna_slider])
    
    # Display initial plot
    update_plot(0, sort_dropdown.value, viz_dropdown.value)
    
    return VBox([controls, output])

# Create and display the dual DNA browser
if pruned_results and simulation_results:
    print("\\n🎛️ Step 5: Creating interactive dual DNA browser...")
    print(f"📊 Browser ready with {len(pruned_results)} pruned DNA vectors")
    print("\\nControls:")
    print("• Sort dropdown: Change ordering of DNAs")
    print("• Show dropdown: Choose between voltage traces, network graph, or both")
    print("• DNA slider: Browse through different DNA solutions")
    print("\\nEach view shows:")
    print("• Voltage traces: Experimental vs control conditions with stimulus markers")
    print("• Network graph: Directed connectivity with edge weights and inhibitory markers")
    print("• Gold borders highlight neurons used for fitness evaluation\\n")
    
    browser = create_dual_dna_browser()
    display(browser)
else:
    print("❌ Cannot create browser: No data available")
    print("\\nCheck:")
    print("1. Results folder path is correct")
    print("2. Score threshold is appropriate")
    print("3. Aggregated results files exist in subfolders")

\n🎛️ Step 5: Creating interactive dual DNA browser...
📊 Browser ready with 2 pruned DNA vectors
\nControls:
• Sort dropdown: Change ordering of DNAs
• Show dropdown: Choose between voltage traces, network graph, or both
• DNA slider: Browse through different DNA solutions
\nEach view shows:
• Voltage traces: Experimental vs control conditions with stimulus markers
• Network graph: Directed connectivity with edge weights and inhibitory markers
• Gold borders highlight neurons used for fitness evaluation\n


## Data Export & Summary

In [18]:
# Save all results for later use
if pruned_results:
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    export_file = f"multiple_ga_analysis_{timestamp}.pkl"
    
    export_data = {
        'config': {
            'results_folder': RESULTS_FOLDER,
            'score_threshold': SCORE_THRESHOLD,
            'pruning_threshold': PRUNING_THRESHOLD,
            'max_dnas_processed': MAX_DNAS_TO_PROCESS
        },
        'high_scoring_dnas': high_scoring_dnas,
        'pruned_results': pruned_results,
        'simulation_results': simulation_results,
        'timestamp': timestamp
    }
    
    with open(export_file, 'wb') as f:
        pickle.dump(export_data, f)
    
    print(f"💾 All results exported to: {export_file}")
    
    # Create summary DataFrame
    summary_data = []
    for i, result in enumerate(pruned_results):
        summary_data.append({
            'DNA_ID': i + 1,
            'Original_Score': result['original_score'],
            'Pruned_Score': result['pruned_score'],
            'Exp_Score': result['final_exp_score'],
            'Cont_Score': result['final_cont_score'],
            'Original_Weights': result['original_nonzero'],
            'Pruned_Weights': result['pruned_nonzero'],
            'Weights_Removed': result['weights_removed'],
            'Reduction_Percent': result['weights_removed'] / result['original_nonzero'] * 100,
            'Run_Folder': result['original_dna']['run_folder'],
            'Generation': result['original_dna']['generation'],
            'Process_ID': result['original_dna']['process_id']
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    print("\n📊 FINAL SUMMARY:")
    print("=" * 60)
    print(f"Results folder analyzed: {RESULTS_FOLDER}")
    print(f"High-scoring DNAs found: {len(high_scoring_dnas)} (threshold: {SCORE_THRESHOLD})")
    print(f"Successfully pruned: {len(pruned_results)}")
    print(f"Average weight reduction: {summary_df['Reduction_Percent'].mean():.1f}%")
    print(f"Best pruned score: {summary_df['Pruned_Score'].max()}")
    print(f"Most efficient (fewest weights): {summary_df['Pruned_Weights'].min()} weights")
    print(f"Total original weights: {summary_df['Original_Weights'].sum()}")
    print(f"Total pruned weights: {summary_df['Pruned_Weights'].sum()}")
    
    # Save summary CSV
    csv_file = f"multiple_ga_summary_{timestamp}.csv"
    summary_df.to_csv(csv_file, index=False)
    print(f"\n📄 Summary table saved to: {csv_file}")
    
    # Display top results
    print("\n🏆 Top 10 Results by Pruned Score:")
    display(summary_df.nlargest(10, 'Pruned_Score')[['DNA_ID', 'Pruned_Score', 'Pruned_Weights', 'Reduction_Percent', 'Run_Folder']])
    
    print("\n🎯 Most Efficient (Fewest Final Weights):")
    display(summary_df.nsmallest(10, 'Pruned_Weights')[['DNA_ID', 'Pruned_Score', 'Pruned_Weights', 'Reduction_Percent', 'Run_Folder']])
    
else:
    print("❌ No results to export")

❌ No results to export
